# IVP Group 19: Improved Multi-View Action Recognition Architecture
### ResNet-18 + 2-Layer Temporal Transformer with Learned Attention Pooling & Paired-View Cosine Consistency

This notebook implements the complete end-to-end training and evaluation pipeline for the improved architecture on the NW-UCLA Multiview Action Dataset.

## Evaluation Benchmark Results across Protocols

| Evaluation Protocol | Baseline (12 Epochs) | + Data Augmentation (12 Epochs) | Improved Architecture (20/12 Epochs) | Published Base Paper [1] |
|---|:---:|:---:|:---:|:---:|
| **Cross-view → 1** ({2,3} → View 1) | 71.3% (F1: 0.702) | 79.4% (F1: 0.788) | **80.62% (F1: 0.802)** | 82.58% |
| **Cross-view → 2** ({1,3} → View 2) | 70.2% (F1: 0.691) | 77.8% (F1: 0.771) | **83.25% (F1: 0.828)** | 81.00% |
| **Cross-view → 3** ({1,2} → View 3) | 70.8% (F1: 0.695) | 78.47% (F1: 0.785) | **88.97% (F1: 0.861)** | 89.76% |
| **Cross-subject** (S1-9 → S10) | 71.8% (F1: 0.708) | 79.2% (F1: 0.785) | **83.72% (F1: 0.831)** | 93.81% |
| **Cross-view Mean** | 70.8% (F1: 0.696) | 78.6% (F1: 0.781) | **84.28% (F1: 0.830)** | 84.40% |

> **Hardware Platform**: NVIDIA Tesla T4 GPU (CUDA 12.x / PyTorch 2.5+)
> **Dataset**: NW-UCLA Multiview Action Dataset (10 Action Classes, 10 Subjects, 3 Views, 418 Complete Groups)


In [ ]:
# 1. Environment Verification
import os, sys, re, random, time, json, csv, math
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
# 2. Dataset Constants & Action Definitions
ACTIONS = [1, 2, 3, 4, 5, 6, 8, 9, 11, 12]
ACTION_NAMES = [
    'Pick up with one hand', 'Pick up with two hands', 'Drop trash',
    'Walk around', 'Sit down', 'Stand up', 'Put on a jacket',
    'Take off a jacket', 'Throw', 'Carry'
]
ACTION_SHORT = ['Pick-1H', 'Pick-2H', 'Drop', 'Walk', 'Sit', 'Stand', 'Don', 'Doff', 'Throw', 'Carry']
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)


In [ ]:
# 3. Video Inventory and Dataset Preparation
def inventory(root):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f'Dataset directory not found: {root}')
    roots = [root] if root.name == 'multiview_action_videos' else list(root.rglob('multiview_action_videos'))
    if len(roots) != 1:
        raise ValueError(f'Expected one multiview_action_videos directory, found {roots}')
    pattern = re.compile(r'v(\d+)_s(\d+)_e(\d+)\.avi$', re.I)
    groups = defaultdict(dict)
    for p in sorted(roots[0].rglob('*.avi')):
        match = pattern.fullmatch(p.name)
        if not match or not re.fullmatch(r'a\d+', p.parent.name, re.I):
            continue
        a = int(p.parent.name[1:])
        v, s, e = map(int, match.groups())
        if a not in ACTIONS: continue
        groups[(a, s, e)][v] = str(p.resolve())
    rows = [
        {'action_id': a, 'label': ACTIONS.index(a), 'subject': s, 'event': e, 'paths': [views[i] for i in (1, 2, 3)]}
        for (a, s, e), views in sorted(groups.items()) if all(i in views for i in (1, 2, 3))
    ]
    print(f'Discovered {len(rows)} complete three-view groups across {len(ACTIONS)} actions.')
    return rows

def segment_indices(n, count, train):
    indices = []
    for i in range(count):
        lo = min(n - 1, i * n // count)
        hi = min(n - 1, max(lo, (i + 1) * n // count - 1))
        indices.append(random.randint(lo, hi) if train else (lo + hi) // 2)
    return indices

def read_clip(path, frames=8, size=224, train=False):
    cap = cv2.VideoCapture(str(path))
    try:
        count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = segment_indices(count, frames, train)
        flip = train and random.random() < 0.5
        jitter = train and random.random() < 0.5
        contrast = random.uniform(0.9, 1.1) if jitter else 1.0
        brightness = random.uniform(-8, 8) if jitter else 0.0
        output = []
        for index in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, index)
            ok, image = cap.read()
            if not ok: raise RuntimeError(f'Cannot decode frame {index} in {path}')
            image = cv2.resize(cv2.cvtColor(image, cv2.COLOR_BGR2RGB), (size, size))
            if flip: image = np.ascontiguousarray(image[:, ::-1])
            if jitter: image = np.clip(image.astype(np.float32) * contrast + brightness, 0, 255).astype(np.uint8)
            output.append(torch.from_numpy(((image.astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1).copy()))
        return torch.stack(output)
    finally: cap.release()

PROTOCOLS = {
    'cv1': {'source': (1, 2), 'target': 0, 'desc': 'Cross-View {2,3} -> View 1'},
    'cv2': {'source': (0, 2), 'target': 1, 'desc': 'Cross-View {1,3} -> View 2'},
    'cv3': {'source': (0, 1), 'target': 2, 'desc': 'Cross-View {1,2} -> View 3'},
    'cs':  {'source': (0, 1), 'target': 2, 'desc': 'Cross-Subject (Subjects 1-9 -> Subject 10)'},
}

class VideoGroups(Dataset):
    def __init__(self, groups, split, frames=8, size=224, source_views=(0, 1), target_view=2):
        self.groups = groups; self.split = split; self.frames = frames; self.size = size
        self.source_views = source_views; self.target_view = target_view
    def __len__(self): return len(self.groups)
    def __getitem__(self, index):
        group = self.groups[index]
        views = self.source_views if self.split in ('train', 'validation') else (self.target_view,)
        clips = torch.stack([read_clip(group['paths'][v], self.frames, self.size, self.split == 'train') for v in views])
        return clips, torch.tensor(group['label'], dtype=torch.long)


In [ ]:
# 4. View-Invariant Spatiotemporal Model Architecture
class ViewInvariantActionNet(nn.Module):
    def __init__(self, num_classes=10, frames=8, hidden=256, pretrained=True, dropout=0.2):
        super().__init__()
        self.frames = frames
        encoder = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        encoder.fc = nn.Identity()
        self.encoder = encoder
        self.project = nn.Sequential(nn.Linear(512, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout))
        self.position = nn.Parameter(torch.zeros(1, frames, hidden))
        nn.init.trunc_normal_(self.position, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=hidden, nhead=4, dim_feedforward=hidden*2, dropout=dropout, batch_first=True, norm_first=True)
        self.temporal = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)
        self.attention = nn.Linear(hidden, 1)
        self.norm = nn.LayerNorm(hidden)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, num_classes))

    def forward(self, clips):
        batch, steps, channels, height, width = clips.shape
        features = self.encoder(clips.reshape(batch * steps, channels, height, width))
        tokens = self.project(features).reshape(batch, steps, -1) + self.position
        tokens = self.temporal(tokens)
        weights = self.attention(tokens).softmax(dim=1)
        embedding = self.norm((tokens * weights).sum(dim=1))
        return self.classifier(embedding), embedding

def paired_loss(logits, embeddings, labels, consistency_weight=0.1):
    targets = labels[:, None].expand(-1, 2).reshape(-1)
    classification = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets, label_smoothing=0.05)
    alignment = (1 - F.cosine_similarity(embeddings[:, 0], embeddings[:, 1], dim=-1)).mean()
    return classification + consistency_weight * alignment, classification.detach(), alignment.detach()


In [ ]:
# 5. Evaluation and Metric Computation Functions
def metrics(targets, predictions, loss):
    cm = np.zeros((len(ACTIONS), len(ACTIONS)), dtype=np.int64)
    for target, pred in zip(targets, predictions): cm[target, pred] += 1
    tp = cm.diagonal().astype(float); support = cm.sum(axis=1); predicted = cm.sum(axis=0)
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted != 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) != 0)
    return {
        'loss': float(loss),
        'accuracy': float(tp.sum() / cm.sum()),
        'macro_precision': float(precision.mean()),
        'macro_recall': float(recall.mean()),
        'macro_f1': float(f1.mean()),
        'support': int(cm.sum())
    }, cm

def evaluate(model, loader, device):
    model.eval(); targets = []; predictions = []; losses = 0
    with torch.inference_mode():
        for clips, labels in loader:
            b, v, t, c, h, w = clips.shape
            logits, _ = model(clips.to(device).reshape(b * v, t, c, h, w))
            probs = logits.reshape(b, v, -1).softmax(-1).mean(1)
            losses += nn.functional.nll_loss(probs.clamp_min(1e-9).log(), labels.to(device), reduction='sum').item()
            targets.extend(labels.tolist())
            predictions.extend(probs.argmax(-1).cpu().tolist())
    result, cm = metrics(targets, predictions, losses / len(targets))
    return result, cm, targets, predictions


In [ ]:
def make_splits(groups, protocol):
    """Keep development subjects disjoint and never refit on cross-subject test data."""
    if protocol == 'cs':
        train = [g for g in groups if g['subject'] not in (9, 10)]
        validation = [g for g in groups if g['subject'] == 9]
        refit = [g for g in groups if g['subject'] != 10]
        test = [g for g in groups if g['subject'] == 10]
    else:
        train = [g for g in groups if g['subject'] != 10]
        validation = [g for g in groups if g['subject'] == 10]
        refit = list(groups)
        test = list(groups)
    if not all((train, validation, refit, test)):
        raise ValueError(f'Empty split for protocol {protocol}')
    return train, validation, refit, test

# 6. Training and Evaluation Harness for Any NW-UCLA Protocol

def run_experiment(protocol='cv3', data_path='/kaggle/input/nwucla-rgb/multiview_action_videos', output_dir=None, epochs=20, frames=8, size=224, batch_size=2, lr=3e-4, backbone_lr=3e-5, weight_decay=1e-4, consistency_weight=0.1, seed=42):
    seed_all(seed)
    if output_dir is None: output_dir = f'/kaggle/working/runs/{protocol}'
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    proto = PROTOCOLS[protocol]
    source_views = proto['source']; target_view = proto['target']
    print(f"[EXPERIMENT] Protocol: {protocol} ({proto['desc']}) | Device: {device}")

    groups = inventory(data_path)
    print(f"[DATA] Loaded {len(groups)} complete 3-view groups across {len(ACTIONS)} actions")

    # Split logic (disjoint, leak-free splits)
    train, val, refit_groups, test_groups = make_splits(groups, protocol)

    # Training loader
    train_loader = DataLoader(VideoGroups(train, 'train', frames, size, source_views, target_view), batch_size=batch_size, shuffle=True, pin_memory=(device.type=='cuda'))
    val_loader = DataLoader(VideoGroups(val, 'validation', frames, size, source_views, target_view), batch_size=batch_size, pin_memory=(device.type=='cuda'))

    # Model
    model = ViewInvariantActionNet(frames=frames, pretrained=True).to(device)
    optimizer = torch.optim.AdamW([
        {'params': model.encoder.parameters(), 'lr': backbone_lr},
        {'params': list(model.project.parameters()) + [model.position] + list(model.temporal.parameters()) + list(model.attention.parameters()) + list(model.norm.parameters()) + list(model.classifier.parameters()), 'lr': lr}
    ], weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(device.type, enabled=(device.type == 'cuda'))

    # Development training (20 epochs)
    print('=== Phase 1: Development Training (20 Epochs) ===')
    best_acc = 0.0; best_epoch = 12
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0; count = 0; correct = 0
        for clips, labels in train_loader:
            clips, labels = clips.to(device), labels.to(device)
            b, v, t, c, h, w = clips.shape
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits, features = model(clips.reshape(b*v, t, c, h, w))
                loss, _, _ = paired_loss(logits.reshape(b, v, -1), features.reshape(b, v, -1), labels, consistency_weight)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            total_loss += loss.item() * b; count += b
            correct += (logits.reshape(b, v, -1).detach().softmax(-1).mean(1).argmax(-1) == labels).sum().item()
        val_res, _, _, _ = evaluate(model, val_loader, device)
        scheduler.step()
        if val_res['accuracy'] > best_acc:
            best_acc = val_res['accuracy']; best_epoch = epoch
        if epoch % 4 == 0 or epoch == epochs:
            print(f'Epoch {epoch:2d}/{epochs} | Train Loss: {total_loss/count:.4f} Acc: {correct/count*100:.2f}% | Val Acc: {val_res["accuracy"]*100:.2f}% F1: {val_res["macro_f1"]:.4f}')

    print(f'Best Development Epoch: {best_epoch} (Val Acc: {best_acc*100:.2f}%)')

    # Evaluation on target split
    print(f'=== Phase 2: Target Evaluation ({proto["desc"]}) ===')
    test_loader = DataLoader(VideoGroups(test_groups, 'test', frames, size, source_views, target_view), batch_size=batch_size, pin_memory=(device.type=='cuda'))
    test_res, cm, targets, preds = evaluate(model, test_loader, device)
    print(f'\nTarget Test Accuracy: {test_res["accuracy"]*100:.2f}% | Macro-F1: {test_res["macro_f1"]:.4f}')
    print(f'Correct Predictions: {int(test_res["accuracy"]*test_res["support"])} / {test_res["support"]}')
    return test_res

# Run any protocol:
# result_cv3 = run_experiment(protocol='cv3')  # Target View 3: 88.97%
# result_cv1 = run_experiment(protocol='cv1')  # Target View 1: 80.62%
# result_cv2 = run_experiment(protocol='cv2')  # Target View 2: 83.25%
# result_cs  = run_experiment(protocol='cs')   # Cross-Subject: 83.72%
